# WEDA-Fall → `arch.snn` (old Colab copy — do not use it for evaluation)

The current pipeline is in the repo: `scripts/learning/train_models.py --arch arch.snn` and `scripts/learning/eval_fall_rules.py`. Full write-up: `docs/pipeline_nhanh_te_nga.tex`.
The cells below are the old pipeline (5 classes, no resample, no event-level rule). Do not use them for evaluation.

What this notebook still mirrors from `health-edge-gateway`:
- load raw GitHub `joaojtmarques/WEDA-FALL` (50 Hz acc+gyro)
- **2 s / 50 Hz** windows (`N, 100, 6`), stride 1 s
- 5 labels: `rest, sit_stand, walk, fall_like, not_worn` (`not_worn` does not occur in WEDA)
- EDA (SMV peak) → clip ±16 g → **12-channel delta spikes** (`Δ=0.05`)
- **subject split** (windows from the same person stay together)
- grid-search `(lr, β)` on val → train SNN LIF `12→32→5` → test (`Se_fall_like`, F1-macro, confusion)

**Runtime → GPU (T4).** `QUICK=True` finishes in a few minutes; `QUICK=False` + `TUNE=True` follows `train_models.py --arch arch.snn`.

In [ ]:
# === SETTINGS (edit this cell, then run the SRC cell) ===
QUICK = True          # True: subsample windows per trial; False = full WEDA
TUNE = False          # True = 4-trial grid as in tune.py (slow)
EPOCHS = 8            # repo default is 8; QUICK can stay at 6
BATCH_SIZE = 32
SEED = 42
DELTA = 0.05          # PDF / preprocess/snn.py
MAX_WINDOWS_PER_TRIAL = 8 if QUICK else None
WEDA_ZIP = "https://github.com/joaojtmarques/WEDA-FALL/archive/refs/heads/main.zip"
DATA_ROOT = "/content/weda_data"

print({"QUICK": QUICK, "TUNE": TUNE, "EPOCHS": EPOCHS, "MAX_WINDOWS_PER_TRIAL": MAX_WINDOWS_PER_TRIAL})

In [ ]:
# === SRC: full pipeline arch.snn / WEDA-Fall (ported from the repo) ===
from __future__ import annotations

import json, os, random, re, shutil, zipfile
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from urllib.request import Request, urlopen

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

SNN_CLASSES = ("rest", "sit_stand", "walk", "fall_like", "not_worn")
WIN, STRIDE, FS = 100, 50, 50
NAME_RE = re.compile(r"U(\d+)_R(\d+)_(accel|gyro|orientation)", re.I)
WEDA_LABEL = {
    "F01": 3, "F02": 3, "F03": 3, "F04": 3, "F05": 3, "F06": 3, "F07": 3, "F08": 3,
    "D01": 2, "D02": 2, "D03": 2, "D07": 2, "D08": 2,
    "D04": 1, "D05": 1, "D06": 1,
    "D09": 0, "D10": 0, "D11": 0,
}


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ----- fetch + prepare (catalog.py / fetch.py / prepare/weda.py) -----
def download_weda(data_root: Path, url: str) -> Path:
    raw = data_root / "raw" / "weda_fall"
    marker = raw / ".fetched"
    if marker.is_file():
        return raw
    raw.mkdir(parents=True, exist_ok=True)
    zpath = raw / "weda_fall.zip"
    print("Downloading", url)
    req = Request(url, headers={"User-Agent": "health-edge-gateway-colab/1.0"})
    with urlopen(req, timeout=180) as resp, zpath.open("wb") as out:
        shutil.copyfileobj(resp, out)
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(raw)
    marker.write_text("github\n", encoding="utf-8")
    return raw


def _find_50hz(root: Path) -> Path | None:
    for cand in [root / "dataset" / "50Hz", root / "50Hz"]:
        if cand.is_dir():
            return cand
    for path in root.rglob("50Hz"):
        if path.is_dir() and any(path.iterdir()):
            return path
    return None


def _group_trials(code_dir: Path):
    grouped = {}
    for csv in code_dir.glob("*.csv"):
        m = NAME_RE.search(csv.stem)
        if not m:
            continue
        user, trial, kind = m.group(1), m.group(2), m.group(3).lower()
        grouped.setdefault((user, trial), {})[kind] = csv
    return grouped


def _read_xyz(path: Path | None):
    if path is None or not path.is_file():
        return None
    raw = np.genfromtxt(path, delimiter=",", dtype=np.float32)
    if raw.ndim == 1:
        raw = raw.reshape(-1, 1)
    if raw.shape[1] >= 4:
        xyz = raw[:, -3:]
    elif raw.shape[1] == 3:
        xyz = raw
    else:
        return None
    return np.nan_to_num(xyz, nan=0.0)


def prepare_weda(raw_root: Path, out_dir: Path, max_windows_per_trial=None) -> Path:
    hz = _find_50hz(raw_root)
    if hz is None:
        raise FileNotFoundError(f"No dataset/50Hz under {raw_root}")
    xs, ys, subjects = [], [], []
    for code_dir in sorted(p for p in hz.iterdir() if p.is_dir()):
        code = code_dir.name.upper()
        if code not in WEDA_LABEL:
            continue
        label = WEDA_LABEL[code]
        for (user, trial), files in _group_trials(code_dir).items():
            acc = _read_xyz(files.get("accel"))
            gyr = _read_xyz(files.get("gyro"))
            if acc is None:
                continue
            if gyr is None or len(gyr) != len(acc):
                gyr = np.zeros_like(acc)
            n = min(len(acc), len(gyr))
            imu = np.concatenate([acc[:n], gyr[:n]], axis=1)
            starts = list(range(0, n - WIN + 1, STRIDE))
            if max_windows_per_trial and len(starts) > max_windows_per_trial:
                idx = np.linspace(0, len(starts) - 1, max_windows_per_trial, dtype=int)
                starts = [starts[i] for i in idx]
            for start in starts:
                xs.append(imu[start : start + WIN])
                ys.append(label)
                subjects.append(f"U{int(user):02d}")
    if not xs:
        raise RuntimeError(f"No WEDA windows from {hz}")
    dest = Path(out_dir)
    if dest.suffix != ".npz":
        dest = dest / "windows.npz"
    dest.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        dest,
        X=np.stack(xs).astype(np.float32),
        y=np.asarray(ys, dtype=np.int64),
        subject=np.asarray(subjects),
        classes=np.asarray(SNN_CLASSES),
    )
    return dest


def maybe_acc_to_g(X: np.ndarray) -> np.ndarray:
    """Fitbit often stores m/s²; this project uses g. Colab only — repo code is unchanged."""
    smv_med = float(np.median(np.sqrt((X[..., :3] ** 2).sum(axis=-1))))
    if smv_med > 3.0:
        X = X.copy()
        X[..., :3] = X[..., :3] / 9.81
        print(f"accel ~ m/s² (median SMV={smv_med:.2f}) → divide by 9.81 to get g")
    else:
        print(f"accel left unchanged (median SMV={smv_med:.2f} g)")
    return X


# ----- preprocess/snn.py -----
def clip_imu_g(windows, acc_clip=16.0, gyr_clip=2000.0):
    out = np.array(windows, dtype=np.float32, copy=True)
    out[..., :3] = np.clip(out[..., :3], -acc_clip, acc_clip)
    out[..., 3:] = np.clip(out[..., 3:], -gyr_clip, gyr_clip)
    return out


def delta_encode(windows, delta=0.05):
    x = np.asarray(windows, dtype=np.float32)
    n, t, c = x.shape
    spikes = np.zeros((n, t, c * 2), dtype=np.float32)
    ref = x[:, 0, :].copy()
    for step in range(t):
        cur = x[:, step, :]
        pos = (cur - ref) > delta
        neg = (ref - cur) > delta
        spikes[:, step, 0::2] = pos
        spikes[:, step, 1::2] = neg
        fire = pos | neg
        ref = np.where(fire, cur, ref)
    return spikes


def prepare_snn(corpus, delta=0.05):
    x = clip_imu_g(corpus["X"])
    out = dict(corpus)
    out["X_raw"] = x
    out["X"] = delta_encode(x, delta)
    out["delta"] = delta
    return out


# ----- split theo subject -----
@dataclass(frozen=True)
class Split:
    X_train: np.ndarray
    y_train: np.ndarray
    X_val: np.ndarray
    y_val: np.ndarray
    X_test: np.ndarray
    y_test: np.ndarray
    subject_train: np.ndarray
    subject_val: np.ndarray
    subject_test: np.ndarray


def subject_holdout(corpus, seed, val_ratio=0.15) -> Split:
    subjects = np.unique(corpus["subject"])
    rng = np.random.default_rng(seed)
    rng.shuffle(subjects)
    n = len(subjects)
    n_test = max(1, n // 5)
    n_val = max(1, int(round((n - n_test) * val_ratio))) if n - n_test > 2 else 1
    if n_test + n_val >= n:
        n_val = 1
        n_test = min(n_test, n - 2)
    test_s = set(subjects[:n_test])
    val_s = set(subjects[n_test : n_test + n_val])
    train_s = set(subjects[n_test + n_val :])
    if not train_s:
        train_s = set(subjects[n_test + n_val - 1 :])
        val_s = set(subjects[n_test : n_test + n_val - 1]) or {subjects[0]}

    def take(pool):
        mask = np.array([s in pool for s in corpus["subject"]], dtype=bool)
        return corpus["X"][mask], corpus["y"][mask], corpus["subject"][mask]

    xt, yt, st = take(train_s)
    xv, yv, sv = take(val_s)
    xe, ye, se = take(test_s)
    return Split(xt, yt, xv, yv, xe, ye, st, sv, se)


def class_weights(y, n_classes=5):
    counts = np.bincount(y.astype(int), minlength=n_classes).astype(np.float64)
    counts = np.maximum(counts, 1.0)
    return (counts.sum() / (n_classes * counts)).astype(np.float32)


# ----- metrics.py -----
def accuracy(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return float((y_true == y_pred).mean()) if len(y_true) else 0.0


def f1_per_class(y_true, y_pred, n_classes=5):
    scores = []
    for k in range(n_classes):
        tp = int(((y_true == k) & (y_pred == k)).sum())
        fp = int(((y_true != k) & (y_pred == k)).sum())
        fn = int(((y_true == k) & (y_pred != k)).sum())
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        scores.append(0.0 if prec + rec == 0 else 2 * prec * rec / (prec + rec))
    return scores


def sensitivity(y_true, y_pred, positive=3):
    mask = y_true == positive
    if not mask.any():
        return 0.0
    return float((y_pred[mask] == positive).mean())


def confusion(y_true, y_pred, n_classes=5):
    mat = np.zeros((n_classes, n_classes), dtype=int)
    for t, p in zip(y_true, y_pred, strict=True):
        mat[int(t), int(p)] += 1
    return mat


# ----- models/snn_lif.py -----
class SurrogateSpike(torch.autograd.Function):
    @staticmethod
    def forward(ctx, membrane, threshold, k):
        ctx.save_for_backward(membrane)
        ctx.threshold = threshold
        ctx.k = k
        return (membrane > threshold).to(membrane.dtype)

    @staticmethod
    def backward(ctx, grad_output):
        (membrane,) = ctx.saved_tensors
        denom = (1.0 + ctx.k * (membrane - ctx.threshold).abs()) ** 2
        return grad_output / denom, None, None


class LIFLayer(nn.Module):
    def __init__(self, n_in, n_out, beta=0.9, threshold=1.0, k=10.0):
        super().__init__()
        self.linear = nn.Linear(n_in, n_out, bias=False)
        nn.init.xavier_uniform_(self.linear.weight, gain=1.0)
        self.beta, self.threshold, self.k = beta, threshold, k

    def step(self, spikes_in, mem, prev_spk):
        mem = self.beta * mem + self.linear(spikes_in) - self.threshold * prev_spk
        spikes = SurrogateSpike.apply(mem, self.threshold, self.k)
        return spikes, mem


class MotionSNN(nn.Module):
    def __init__(self, n_in=12, n_hidden=32, n_out=5, beta=0.9):
        super().__init__()
        self.hidden = LIFLayer(n_in, n_hidden, beta=beta)
        self.output = LIFLayer(n_hidden, n_out, beta=beta)
        self.n_hidden, self.n_out = n_hidden, n_out

    def forward(self, spikes):
        batch, steps, _ = spikes.shape
        mem_h = spikes.new_zeros(batch, self.n_hidden)
        mem_o = spikes.new_zeros(batch, self.n_out)
        spk_h = spikes.new_zeros(batch, self.n_hidden)
        spk_o = spikes.new_zeros(batch, self.n_out)
        counts = spikes.new_zeros(batch, self.n_out)
        for t in range(steps):
            spk_h, mem_h = self.hidden.step(spikes[:, t], mem_h, spk_h)
            spk_o, mem_o = self.output.step(spk_h, mem_o, spk_o)
            counts = counts + spk_o
        return counts


@torch.no_grad()
def snn_logits(model, x, device, batch=128):
    model.eval()
    chunks = []
    for i in range(0, len(x), batch):
        xb = torch.tensor(x[i : i + batch], dtype=torch.float32, device=device)
        chunks.append(model(xb).cpu().numpy())
    return np.concatenate(chunks, axis=0) if chunks else np.zeros((0, 5), np.float32)


def eval_snn(model, x, y, device):
    logits = snn_logits(model, x, device)
    pred = logits.argmax(axis=1)
    f1s = f1_per_class(y, pred, 5)
    return {
        "accuracy": accuracy(y, pred),
        "f1_macro": float(np.mean(f1s)),
        "f1_per_class": f1s,
        "sensitivity_fall_like": sensitivity(y, pred, 3),
        "pred": pred,
    }


def fit_snn(x, y, *, epochs, batch_size, lr, beta, device, x_val=None, y_val=None):
    model = MotionSNN(beta=beta).to(device)
    weights = torch.tensor(class_weights(y, 5), device=device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss(weight=weights)
    loader = DataLoader(
        TensorDataset(torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)),
        batch_size=batch_size,
        shuffle=True,
    )
    history, best, best_score = [], None, -1.0
    for epoch in range(1, epochs + 1):
        model.train()
        total = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
            total += float(loss.detach()) * len(yb)
        row = {"epoch": epoch, "train_loss": total / max(1, len(y))}
        if x_val is not None:
            metrics = eval_snn(model, x_val, y_val, device)
            row.update({f"val_{k}": v for k, v in metrics.items() if k != "pred"})
            if metrics["f1_macro"] >= best_score:
                best_score = metrics["f1_macro"]
                best = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        history.append(row)
        print(
            f"  epoch {epoch:02d}  loss={row['train_loss']:.4f}"
            + (f"  val_f1={row.get('val_f1_macro', 0):.3f}  Se_fall={row.get('val_sensitivity_fall_like', 0):.3f}" if x_val is not None else "")
        )
    if best is not None:
        model.load_state_dict(best)
    return model, history


def tune_snn(split, epochs, batch_size, device, max_trials=4):
    grid = [
        {"lr": 1e-3, "beta": 0.9},
        {"lr": 5e-4, "beta": 0.85},
        {"lr": 2e-3, "beta": 0.95},
        {"lr": 1e-3, "beta": 0.8},
    ][:max_trials]
    best, best_score, trials = grid[0], -1.0, []
    for trial in grid:
        print("tune", trial)
        model, _ = fit_snn(
            split.X_train, split.y_train,
            epochs=max(3, epochs // 2), batch_size=batch_size,
            lr=trial["lr"], beta=trial["beta"], device=device,
            x_val=split.X_val, y_val=split.y_val,
        )
        m = eval_snn(model, split.X_val, split.y_val, device)
        score = 0.5 * m["f1_macro"] + 0.5 * m["sensitivity_fall_like"]
        trials.append({**trial, "f1_macro": m["f1_macro"], "sensitivity_fall_like": m["sensitivity_fall_like"], "score": score})
        print("  score", round(score, 4))
        if score > best_score:
            best, best_score = trial, score
    return {"best": best, "trials": trials}


def eda_snn(corpus):
    y = corpus["y"]
    names = list(SNN_CLASSES)
    counts = {names[i]: int((y == i).sum()) for i in range(len(names))}
    x = corpus.get("X_raw", corpus["X"])
    smv = np.sqrt((x[..., :3] ** 2).sum(axis=-1)).max(axis=1) if x.ndim == 3 else None
    by_class = {}
    for i, name in enumerate(names):
        mask = y == i
        if smv is not None and mask.any():
            by_class[name] = {
                "n": int(mask.sum()),
                "smv_peak_mean": float(smv[mask].mean()),
                "smv_peak_p95": float(np.quantile(smv[mask], 0.95)),
            }
        else:
            by_class[name] = {"n": int(mask.sum())}
    present = [v for v in counts.values() if v > 0]
    return {
        "source": corpus.get("source"),
        "n_windows": int(len(y)),
        "n_subjects": int(len(np.unique(corpus["subject"]))),
        "class_counts": counts,
        "imbalance_ratio": float(max(present) / max(1, min(present))) if present else 0.0,
        "per_class": by_class,
        "notes": [
            "2 s windows at 50 Hz; split by subject, never by window.",
            "not_worn is absent on WEDA-Fall (count=0).",
            "Every window in an F* trial is labeled fall_like, including the samples before impact.",
        ],
    }


def plot_eda(eda, history, cm, class_names):
    if plt is None:
        return
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    names = list(eda["class_counts"])
    axes[0].bar(names, [eda["class_counts"][n] for n in names])
    axes[0].set_title("Windows per class")
    axes[0].tick_params(axis="x", rotation=30)
    means = [eda["per_class"][n].get("smv_peak_mean", 0) for n in names]
    axes[1].bar(names, means)
    axes[1].set_title("SMV peak mean (g)")
    axes[1].tick_params(axis="x", rotation=30)
    im = axes[2].imshow(cm, cmap="Blues")
    axes[2].set_xticks(range(len(class_names)))
    axes[2].set_yticks(range(len(class_names)))
    axes[2].set_xticklabels(class_names, rotation=30, ha="right")
    axes[2].set_yticklabels(class_names)
    axes[2].set_title("Confusion test (true × pred)")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            axes[2].text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=axes[2], fraction=0.046)
    plt.tight_layout()
    plt.show()
    if history:
        fig2, ax = plt.subplots(figsize=(6, 3))
        ax.plot([r["epoch"] for r in history], [r["train_loss"] for r in history], label="train_loss")
        if "val_f1_macro" in history[0]:
            ax.plot([r["epoch"] for r in history], [r["val_f1_macro"] for r in history], label="val_f1_macro")
        ax.legend()
        ax.set_xlabel("epoch")
        plt.tight_layout()
        plt.show()


def run_pipeline():
    set_seed(SEED)
    root = Path(DATA_ROOT)
    tag = f"q{int(QUICK)}_w{MAX_WINDOWS_PER_TRIAL}"
    npz = root / "weda_fall" / f"windows_{tag}.npz"
    if not npz.is_file():
        raw = download_weda(root, WEDA_ZIP)
        npz = prepare_weda(raw, npz, MAX_WINDOWS_PER_TRIAL)
    blob = np.load(npz, allow_pickle=True)
    raw_corpus = {
        "X": maybe_acc_to_g(blob["X"].astype(np.float32)),
        "y": blob["y"].astype(np.int64),
        "subject": blob["subject"].astype(str),
        "source": "weda_fall",
        "fs": FS,
    }
    print("loaded", raw_corpus["X"].shape, "subjects", len(np.unique(raw_corpus["subject"])))

    prepared = prepare_snn(raw_corpus, delta=DELTA)
    eda = eda_snn(prepared)
    print(json.dumps({k: eda[k] for k in ("n_windows", "n_subjects", "class_counts", "imbalance_ratio")}, indent=2))
    print("per_class SMV:", json.dumps(eda["per_class"], indent=2))
    for n in eda["notes"]:
        print("-", n)

    split = subject_holdout(prepared, seed=SEED)
    print("split", {
        "n_train": len(split.y_train), "n_val": len(split.y_val), "n_test": len(split.y_test),
        "subjects_train": sorted(set(split.subject_train.tolist())),
        "subjects_val": sorted(set(split.subject_val.tolist())),
        "subjects_test": sorted(set(split.subject_test.tolist())),
        "rule": "split_by_subject_no_window_leakage",
    })

    tuned = tune_snn(split, EPOCHS, BATCH_SIZE, DEVICE) if TUNE else {"best": {"lr": 1e-3, "beta": 0.9}, "trials": []}
    best = tuned["best"]
    print("hparams", best)

    model, history = fit_snn(
        split.X_train, split.y_train,
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        lr=best["lr"], beta=best["beta"], device=DEVICE,
        x_val=split.X_val, y_val=split.y_val,
    )
    test = eval_snn(model, split.X_test, split.y_test, DEVICE)
    pred = test.pop("pred")
    cm = confusion(split.y_test, pred, 5)
    test["confusion"] = cm.tolist()
    test["classes"] = list(SNN_CLASSES)
    test["n_test"] = int(len(split.y_test))
    print("TEST", json.dumps({k: v for k, v in test.items() if k != "confusion"}, indent=2))
    print("confusion rows=true", SNN_CLASSES)
    print(cm)

    ckpt = Path("/content/arch_snn_weda.pt")
    torch.save({"architecture": "arch.snn", "state_dict": model.state_dict(), "hparams": best}, ckpt)
    print("saved", ckpt)

    plot_eda(eda, history, cm, SNN_CLASSES)
    return {"eda": eda, "tune": tuned, "test": test, "history": history, "model": model, "split": split}


RESULT = run_pipeline()